#### RQ1: How does the logical consistency of the CBM change when introducing the requirements via Fuzzy Loss

- This RQ aims to evaluate the logical consistency of the CBM and validate the claim that the FuzzyLoss CBM learned the rules and adheres to them on the GTSRB in distribution dataset.

In [1]:
# imports
import sys
from pathlib import Path
import torch

# changing the cwd to be source for all the imports to continue working
current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(f"{parent_dir}/src"))

from models.architectures import CBMSequentialEfficientNetFCN
from train_cbm import cbm_load_config
from models.trainer.cbm_trainer import CBMTrainer
from rule_eval import construct_full_graph

# Import utility functions
from analysis_utils import (
    get_dataset_predictions,
    analyze_fuzzy_loss_single_model,
    compare_fuzzy_losses,
    analyze_rule_violations,
    compare_violations,
    print_fuzzy_loss_results,
    print_violation_results,
)

In [2]:
# model configs and model loading
baseline_cbm_config = cbm_load_config(Path("../files/configs/GTSRB_CBM_config_loading.yaml"))
baseline_cbm = CBMSequentialEfficientNetFCN(baseline_cbm_config)

fuzzy_cbm_config = cbm_load_config(Path("../files/configs/GTSRB_CBM_config_best_trial_loading.yaml"))
fuzzy_cbm = CBMSequentialEfficientNetFCN(fuzzy_cbm_config)

Directory 'experiments/20251022_173341' created successfully.
Directory 'experiments/20251022_173341' created successfully.


In [3]:
# model paths for loading models
baseline_cbm_concept_predictor_path = Path("../notebooks/best_acc_models/20251016_224601_s907_baseline_concept_predictor_best_model.pt")
baseline_cbm_label_predictor_path = Path("../experiments/baseline_cbm/models/20251001_083717_label_predictor_best_model.pt")
fuzzy_cbm_concept_predictor_path = Path("../notebooks/best_acc_models/20251020_223819_s269_concept_predictor_best_model.pt")
fuzzy_cbm_label_precitor_path = Path("../experiments/fuzzy_CBM/models/20251001_113637_label_predictor_best_model.pt")

In [4]:
# Load the baseline model components weights
baseline_cbm.concept_predictor.load_state_dict(
    torch.load(baseline_cbm_concept_predictor_path, map_location=baseline_cbm_config.device, weights_only=True)
)
baseline_cbm.label_predictor.load_state_dict(
    torch.load(baseline_cbm_label_predictor_path, map_location=baseline_cbm_config.device, weights_only=True)
)

# Load the fuzzy model components weights
fuzzy_cbm.concept_predictor.load_state_dict(
    torch.load(fuzzy_cbm_concept_predictor_path, map_location=fuzzy_cbm_config.device, weights_only=True)
)
fuzzy_cbm.label_predictor.load_state_dict(
    torch.load(fuzzy_cbm_label_precitor_path, map_location=fuzzy_cbm_config.device, weights_only=True)
)

# Set models to evaluation mode
baseline_cbm.eval()
fuzzy_cbm.eval()

print(f"  Baseline CBM: {baseline_cbm_concept_predictor_path.parent}")
print(f"  Fuzzy CBM:    {fuzzy_cbm_concept_predictor_path.parent}")

  Baseline CBM: ../notebooks/best_acc_models
  Fuzzy CBM:    ../notebooks/best_acc_models


In [5]:
# Load GTSRB Dataset
dataset_factory = baseline_cbm_config.dataset.factory(
    seed=baseline_cbm_config.seed, config=baseline_cbm_config.dataset
).set_dataloaders()

train_loader = dataset_factory.train_dataloader
val_loader = dataset_factory.val_dataloader
test_loader = dataset_factory.test_dataloader

print(f"  Train samples: {len(dataset_factory.train_dataset)}")
print(f"  Val samples:   {len(dataset_factory.val_dataset)}")
print(f"  Test samples:  {len(dataset_factory.test_dataset)}")

  Train samples: 31368
  Val samples:   7841
  Test samples:  12630


In [6]:
# Setup Trainers and Get Fuzzy Loss Function
baseline_cbm_trainer = CBMTrainer(
    config=baseline_cbm_config,
    model=baseline_cbm,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)

fuzzy_cbm_trainer = CBMTrainer(
    config=fuzzy_cbm_config,
    model=fuzzy_cbm,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)

# Get the fuzzy loss function
neutral_fuzzy_loss = fuzzy_cbm_trainer.concept_predictor_trainer.criterion

In [7]:
# Load rule checker
rule_checker = construct_full_graph()#

In [8]:
# Cell: GTSRB Analysis (In-Distribution)
print("="*80)
print("ANALYZING GTSRB DATASET (In-Distribution)")
print("="*80)

# Get predictions from both models
print("\nGetting predictions from both models...")
baseline_preds_gtsrb = get_dataset_predictions(
    baseline_cbm, test_loader, baseline_cbm_config.device, "GTSRB (Baseline)"
)
fuzzy_preds_gtsrb = get_dataset_predictions(
    fuzzy_cbm, test_loader, fuzzy_cbm_config.device, "GTSRB (Fuzzy)"
)

# ============================================================================
# FUZZY LOSS ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("FUZZY LOSS ANALYSIS - GTSRB")
print("="*80)

baseline_fuzzy_gtsrb = analyze_fuzzy_loss_single_model(
    baseline_preds_gtsrb['logits'], 
    baseline_preds_gtsrb['predictions'],
    neutral_fuzzy_loss, 
    'Baseline CBM', 
    'GTSRB'
)

fuzzy_fuzzy_gtsrb = analyze_fuzzy_loss_single_model(
    fuzzy_preds_gtsrb['logits'], 
    fuzzy_preds_gtsrb['predictions'],
    neutral_fuzzy_loss, 
    'Fuzzy CBM', 
    'GTSRB'
)

fuzzy_comparison_gtsrb = compare_fuzzy_losses(baseline_fuzzy_gtsrb, fuzzy_fuzzy_gtsrb)

print_fuzzy_loss_results(baseline_fuzzy_gtsrb)
print_fuzzy_loss_results(fuzzy_fuzzy_gtsrb, fuzzy_comparison_gtsrb)

# Display per-rule comparison table
print("\n" + "="*80)
print("PER-RULE FUZZY LOSS COMPARISON")
print("="*80)
print("\n" + fuzzy_comparison_gtsrb['rule_comparison'].to_string(index=False))

# ============================================================================
# RULE VIOLATION ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("RULE VIOLATION ANALYSIS - GTSRB")
print("="*80)

baseline_viols_gtsrb = analyze_rule_violations(
    baseline_preds_gtsrb['predictions'], 
    'GTSRB', 
    'Baseline CBM', 
    rule_checker
)

fuzzy_viols_gtsrb = analyze_rule_violations(
    fuzzy_preds_gtsrb['predictions'], 
    'GTSRB', 
    'Fuzzy CBM', 
    rule_checker
)

violation_comparison_gtsrb = compare_violations(baseline_viols_gtsrb, fuzzy_viols_gtsrb)

print_violation_results(baseline_viols_gtsrb)
print_violation_results(fuzzy_viols_gtsrb, violation_comparison_gtsrb)

# Display per-constraint comparison table
print("\n" + "="*80)
print("PER-CONSTRAINT VIOLATION COMPARISON")
print("="*80)
print("\n" + violation_comparison_gtsrb['constraint_comparison'].to_string(index=False))

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("GTSRB ANALYSIS SUMMARY")
print("="*80)

print(f"\nLogical Consistency Improvements:")
print(f"  Fuzzy Loss Reduction:     {fuzzy_comparison_gtsrb['relative_reduction_pct']:.2f}%")
print(f"  Violation Rate Reduction: {violation_comparison_gtsrb['relative_improvement_pct']:.2f}%")

print(f"\nKey Metrics:")
print(f"  Baseline CBM:")
print(f"    - Fuzzy Loss:      {baseline_fuzzy_gtsrb['fuzzy_loss']:.10f}")
print(f"    - Violation Rate:  {baseline_viols_gtsrb['violation_rate']:.2f}%")
print(f"  Fuzzy CBM:")
print(f"    - Fuzzy Loss:      {fuzzy_fuzzy_gtsrb['fuzzy_loss']:.10f}")
print(f"    - Violation Rate:  {fuzzy_viols_gtsrb['violation_rate']:.2f}%")

if fuzzy_comparison_gtsrb['relative_reduction_pct'] > 0 and violation_comparison_gtsrb['relative_improvement_pct'] > 0:
    print(f"\n✓ Fuzzy CBM shows improved logical consistency on GTSRB")
    print(f"  Both fuzzy loss and violation rate metrics confirm the improvement")
else:
    print(f"\n⚠ Mixed results - review individual metrics above")

ANALYZING GTSRB DATASET (In-Distribution)

Getting predictions from both models...


Getting GTSRB (Fuzzy) predictions: 100%|██████████| 99/99 [00:12<00:00,  7.63it/s]



FUZZY LOSS ANALYSIS - GTSRB

Baseline CBM on GTSRB:
  Standard BCE Loss:    0.0003751071
  Fuzzy Rules Loss:     0.0137938075
  Total Loss:           0.0141689146

Fuzzy CBM on GTSRB:
  Standard BCE Loss:    0.0002976162
  Fuzzy Rules Loss:     0.0130780460
  Total Loss:           0.0133756623

  Improvement over Baseline:
    Absolute: 0.0007157614
    Relative: 5.19%

PER-RULE FUZZY LOSS COMPARISON

                          Rule  Baseline Loss  Fuzzy CBM Loss   Improvement  Improvement %
     at_most_one_border_colour   1.156950e-05    0.000000e+00  1.156950e-05     100.000000
no_symbols_exactly_two_colours   8.708388e-04    3.486725e-04  5.221663e-04      59.961305
           at_most_one_warning   1.229602e-02    1.211789e-02  1.781275e-04       1.448660
      warning_sign_exclusivity   4.653594e-04    4.614065e-04  3.952882e-06       0.849426
             exactly_one_shape   7.501225e-05    7.501248e-05 -2.328306e-10      -0.000310
       exactly_one_main_colour   7.500465e-05   